<a href="https://colab.research.google.com/github/dennisgathu8/36CHAMBERS/blob/main/Points.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
from google.colab import files
from io import BytesIO
from openpyxl import load_workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side

# Step 1: Upload Excel file
print("Please upload your Excel file")
uploaded = files.upload()

# Step 2: Load the uploaded Excel file into a pandas DataFrame
for file_name in uploaded.keys():
    df = pd.read_excel(BytesIO(uploaded[file_name]))

# Capitalize the column names for the Data sheet
df.columns = df.columns.str.capitalize()

# Step 3: Define the function to calculate points based on deposit amount
def calculate_points(deposit):
    if deposit >= 10000:
        return 1000
    elif 5000 <= deposit <= 9999:
        return 500
    elif 100 <= deposit <= 4999:
        return 100
    elif 50 <= deposit <= 99:
        return 50
    elif 10 <= deposit <= 49:
        return 10
    else:
        return 0

# Step 4: Apply the function to add the 'points' column
df['Points'] = df['Total_amount'].apply(calculate_points)

# Step 5: Create summary based on deposit ranges
summary_data = {
    'Tiers': [
        'Above 10,000 ETB',
        '5,000 ETB to 9,999 ETB',
        '100 ETB to 4,999 ETB',
        '50 ETB to 99 ETB',
        '10 ETB to 49 ETB'
    ],
    'Number of Users': [
        (df['Total_amount'] >= 10000).sum(),
        df[(df['Total_amount'] >= 5000) & (df['Total_amount'] <= 9999)].shape[0],
        df[(df['Total_amount'] >= 100) & (df['Total_amount'] <= 4999)].shape[0],
        df[(df['Total_amount'] >= 50) & (df['Total_amount'] <= 99)].shape[0],
        df[(df['Total_amount'] >= 10) & (df['Total_amount'] <= 49)].shape[0]
    ],
    'Reward': [
        1000,
        500,
        100,
        50,
        10
    ],
    'Total Points Awarded': [
        df[df['Total_amount'] >= 10000]['Points'].sum(),
        df[(df['Total_amount'] >= 5000) & (df['Total_amount'] <= 9999)]['Points'].sum(),
        df[(df['Total_amount'] >= 100) & (df['Total_amount'] <= 4999)]['Points'].sum(),
        df[(df['Total_amount'] >= 50) & (df['Total_amount'] <= 99)]['Points'].sum(),
        df[(df['Total_amount'] >= 10) & (df['Total_amount'] <= 49)]['Points'].sum()
    ],
    'Total Deposits': [
        df[df['Total_amount'] >= 10000]['Total_amount'].sum(),
        df[(df['Total_amount'] >= 5000) & (df['Total_amount'] <= 9999)]['Total_amount'].sum(),
        df[(df['Total_amount'] >= 100) & (df['Total_amount'] <= 4999)]['Total_amount'].sum(),
        df[(df['Total_amount'] >= 50) & (df['Total_amount'] <= 99)]['Total_amount'].sum(),
        df[(df['Total_amount'] >= 10) & (df['Total_amount'] <= 49)]['Total_amount'].sum()
    ]
}

# Step 6: Add a row for totals
total_users = sum(summary_data['Number of Users'])
total_points = sum(summary_data['Total Points Awarded'])
total_deposits = sum(summary_data['Total Deposits'])

summary_data['Tiers'].append('Total')
summary_data['Number of Users'].append(total_users)
summary_data['Reward'].append('')
summary_data['Total Points Awarded'].append(total_points)
summary_data['Total Deposits'].append(total_deposits)

# Step 7: Create DataFrame
summary_df = pd.DataFrame(summary_data)

# Step 8: Save the updated DataFrame and summary to a new Excel file with multiple sheets
with pd.ExcelWriter('formatted_summary.xlsx', engine='openpyxl') as writer:
    df.to_excel(writer, sheet_name='Data', index=False)  # Save the original data with points
    summary_df.to_excel(writer, sheet_name='Summary', index=False)  # Save the summary

# Step 9: Load workbook and get access to both sheets
wb = load_workbook('formatted_summary.xlsx')
ws_data = wb['Data']
ws_summary = wb['Summary']

# Step 10: Apply formatting to both sheets

# Common formatting
header_font = Font(bold=True, size=12, color='FFFFFF')  # Set header font to 12 and bold
header_fill = PatternFill(start_color='4F81BD', end_color='4F81BD', fill_type='solid')
data_font = Font(size=11)
alignment = Alignment(horizontal="center", vertical="center")
thin_border = Border(left=Side(style='thin'), right=Side(style='thin'), top=Side(style='thin'), bottom=Side(style='thin'))

# Define colors for points in the data
colors = {
    1000: 'ADD8E6',  # Light Blue
    500: '90EE90',   # Light Green
    100: 'FFD700',   # Gold
    50: 'FFB6C1',    # Light Pink
    10: 'FFDEAD',    # NavajoWhite
}

# Auto-adjust column widths based on content
for ws in [ws_data, ws_summary]:
    for col in ws.columns:
        max_length = 0
        column = col[0].column_letter  # Get the column name (A, B, C, etc.)
        for cell in col:
            try:  # Check the length of the content of each cell
                if len(str(cell.value)) > max_length:
                    max_length = len(str(cell.value))
            except:
                pass
        adjusted_width = max_length + 2
        ws.column_dimensions[column].width = adjusted_width  # Adjust column width

# Apply header formatting for both sheets
for sheet in [ws_data, ws_summary]:
    for cell in sheet[1]:
        cell.font = header_font
        cell.fill = header_fill
        cell.alignment = alignment
        cell.border = thin_border

# Apply row formatting based on points in the Data sheet
for row in ws_data.iter_rows(min_row=2, max_row=ws_data.max_row, min_col=1, max_col=ws_data.max_column):
    points_value = row[df.columns.get_loc('Points')].value
    if points_value in colors:
        fill_color = PatternFill(start_color=colors[points_value], end_color=colors[points_value], fill_type='solid')
        for cell in row:
            cell.fill = fill_color
            cell.font = data_font
            cell.alignment = alignment
            cell.border = thin_border

# Apply formatting to the Summary sheet based on the "Reward" column
for row in ws_summary.iter_rows(min_row=2, max_row=ws_summary.max_row, min_col=1, max_col=ws_summary.max_column):
    reward_value = row[summary_df.columns.get_loc('Reward')].value
    if reward_value in colors:
        fill_color = PatternFill(start_color=colors[reward_value], end_color=colors[reward_value], fill_type='solid')
        for cell in row:
            cell.fill = fill_color
            cell.font = data_font
            cell.alignment = alignment
            cell.border = thin_border

# Apply borders to the "Total" row in the Summary sheet
for cell in ws_summary[ws_summary.max_row]:
    cell.border = thin_border

# Apply phone column formatting (no commas or decimals)
phone_column_index = df.columns.get_loc('Phone') + 1  # Assuming 'Phone' is the column name
for row in ws_data.iter_rows(min_row=2, max_row=ws_data.max_row, min_col=1, max_col=ws_data.max_column):
    if row[phone_column_index - 1].value:
        row[phone_column_index - 1].number_format = '0'

# Apply commas to all numeric columns except the phone and user ID columns
user_id_column_index = df.columns.get_loc('User_id') + 1  # Assuming 'User_id' is the column name
for ws in [ws_data, ws_summary]:
    for row in ws.iter_rows(min_row=2, max_row=ws.max_row, min_col=1, max_col=ws.max_column):
        for cell in row:
            if isinstance(cell.value, (int, float)) and cell.column not in [phone_column_index, user_id_column_index]:
                cell.number_format = '#,##0'  # Add commas for thousands separator

# Step 11: Save and download the updated file
wb.save('formatted_summary.xlsx')
files.download('formatted_summary.xlsx')


Please upload your Excel file


Saving aradapoints20240918.xlsx to aradapoints20240918.xlsx


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>